In [ ]:
import time
from tqdm import notebook

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import (
    Dataset, 
    DataLoader,
    RandomSampler,
    Sampler,
    SequentialSampler,
    SubsetRandomSampler
)
import transformers
import warnings

In [4]:
RANDOM_STATE = 654321
TEST_SIZE = 0.30

OUTPUT_PATH = './'

DEVICE = (torch.device('mps') if torch.backends.mps.is_available()
          else torch.device('cpu'))
print(f'Training on device {DEVICE}')

CONFIG_PATH = './rubert_cased_L-12_H-768_A-12_pt_v1/config.json'
MODEL_PATH = './rubert_cased_L-12_H-768_A-12_pt_v1/pytorch_model.bin'
VOCAB_PATH = './rubert_cased_L-12_H-768_A-12_pt_v1/vocab_new.txt'

Training on device mps


In [5]:
config = transformers.BertConfig.from_json_file(CONFIG_PATH)
tokenizer = transformers.BertTokenizer(vocab_file=VOCAB_PATH)
model = transformers.BertModel.from_pretrained(
    MODEL_PATH, 
    config=config,
    ignore_mismatched_sizes=True
)

In [4]:
model.encoder.layer

ModuleList(
  (0-11): 12 x BertLayer(
    (attention): BertAttention(
      (self): BertSdpaSelfAttention(
        (query): Linear(in_features=768, out_features=768, bias=True)
        (key): Linear(in_features=768, out_features=768, bias=True)
        (value): Linear(in_features=768, out_features=768, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (output): BertSelfOutput(
        (dense): Linear(in_features=768, out_features=768, bias=True)
        (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
    )
    (intermediate): BertIntermediate(
      (dense): Linear(in_features=768, out_features=3072, bias=True)
      (intermediate_act_fn): GELUActivation()
    )
    (output): BertOutput(
      (dense): Linear(in_features=3072, out_features=768, bias=True)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
  )
)

In [18]:

for i, param in enumerate(model.embeddings.parameters()):
    if i == 0:
        print(f'    weights.requires_grad = {param.requires_grad}')
    else:
        print(f'    bias.requires_grad = {param.requires_grad}', end='\n\n')

    weights.requires_grad = True
    bias.requires_grad = True

    bias.requires_grad = True

    bias.requires_grad = True

    bias.requires_grad = True



In [6]:
class OperationsBERT(nn.Module):
    def __init__(self, bert_model):
        super().__init__()
        self.bert_module = bert_model
        self.dropout = nn.Dropout(0.1)

        # Заморозка всех параметров модели
        # for param in self.bert_module.parameters():
        #    param.requires_grad = False

        # Раскомментируйте, если хотите лишь переобучить определённые слои.
        self.bert_module.requires_grad_(False)

        # Разморозка параметров слоя эмбеддингов
        for  param in self.bert_module.embeddings.parameters():
            param.requires_grad = True 
        
        # for layer in self.bert_module.encoder.layer[11:]:
        #    for param in layer.parameters():
        #        param.requires_grad = True
        for param in self.bert_module.pooler.parameters():
            param.requires_grad = True

        self.final = nn.Linear(in_features=768, out_features=2, bias=True)
        
    def forward(self, inputs):
        ids, mask, token_type_ids = inputs['ids'], inputs['mask'], inputs['token_type_ids']
        # print(ids.size(), mask.size(), token_type_ids.size())
        x = self.bert_module(ids, mask, token_type_ids)
        x = self.dropout(x['pooler_output'])
        out = self.final(x)
        return out

In [7]:
class BertDataset(Dataset):
    def __init__(self, df, tokenizer, max_length=512):
        super(BertDataset, self).__init__()
        self.df=df
        self.tokenizer=tokenizer
        self.target=self.df['target']
        self.max_length=max_length
        
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        
        X = self.df['echo_from_epikriz'].values[idx]
        y = self.target.values[idx]
        
        inputs = self.tokenizer.encode_plus(
            X,
            pad_to_max_length=True,
            add_special_tokens=True,
            return_attention_mask=True,
            max_length=self.max_length,
        )
        ids = inputs['input_ids']
        token_type_ids = inputs['token_type_ids']
        mask = inputs['attention_mask']

        x = {
            'ids': torch.tensor(ids, dtype=torch.long).to(DEVICE),
            'mask': torch.tensor(mask, dtype=torch.long).to(DEVICE),
            'token_type_ids': torch.tensor(token_type_ids, dtype=torch.long).to(DEVICE)
            }
        y = torch.tensor(y, dtype=torch.long).to(DEVICE)
        
        return x, y

In [8]:
def f1_macro(final_outputs, final_targets):
    final_outputs = np.array(final_outputs)
    final_targets = np.array(final_targets)
    final_targets = np.reshape(final_targets, (-1,))

    TP = ((final_outputs == 1) & (final_targets == 1)).sum().item()
    FP = ((final_outputs == 1) & (final_targets == 0)).sum().item()
    FN = ((final_outputs == 0) & (final_targets == 1)).sum().item()
    TN = ((final_outputs == 0) & (final_targets == 0)).sum().item()
    
    precision_1 = TP / (TP + FP) if TP + FP > 0 else 0
    recall_1 = TP / (TP + FN) if TP + FN > 0 else 0
    
    f1_1 = (
        2 * (precision_1 * recall_1) / (precision_1 + recall_1) 
        if (precision_1 + recall_1) > 0 else 0
        )
    
    precision_0 = TN / (TN + FN) if TN + FN > 0 else 0
    recall_0 = TN / (TN + FP) if TN + FP > 0 else 0
    
    f1_0 = (
        2 * (precision_0 * recall_0) / (precision_0 + recall_0) 
        if (precision_0 + recall_0) > 0 else 0
        )
    
    f1_macro = (f1_1 + f1_0) / 2

    return f1_macro, TP, FP, FN, TN

In [9]:
def train(epoch, model, dataloader, loss_fn, f1_macro, optimizer, max_steps=None):
    model.train()
    total_acc, total_count = 0, 0
    log_interval = 50
    start_time = time.time()
    final_targets = []
    final_outputs = []

    for idx, (inputs, label) in enumerate(dataloader):
        predicted_label = model(inputs)
        loss = loss_fn(predicted_label, torch.eye(2).to(DEVICE)[label])
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        
        total_acc += (predicted_label.argmax(1) == label).sum().item()
        total_count += label.size(0)

        preds = predicted_label.argmax(1)
        targets = label.detach().cpu().numpy().tolist()
        output = preds.detach().cpu().numpy().tolist()
        final_targets.extend(targets)
        final_outputs.extend(output)
    

        if idx % log_interval == 0:
            elapsed = time.time() - start_time
            print(
                'Epoch {:3d} | {:5d}/{:5d} batches '
                '| accuracy {:8.3f} | loss {:8.3f} ({:.3f}s)'.format(
                    epoch, idx, len(dataloader), total_acc / total_count, loss.item(), elapsed
                )
            )
            # total_acc, total_count = 0, 0
            start_time = time.time()

        if max_steps is not None:
            if idx == max_steps:
                return {'loss': loss.item(), 'acc': total_acc / total_count}
            
    f1_macro, TP, FP, FN, TN = f1_macro(final_outputs, final_targets)
    
    return {
        'loss': loss.item(), 
        'acc': total_acc / total_count,
        'f1': f1_macro, 
        'TP': TP,
        'FP': FP,
        'TN': TN,
        'FN': FN
        }


def evaluate(model, dataloader, loss_fn, f1_macro):
    model.eval()
    total_acc, total_count = 0, 0

    with torch.no_grad():
        final_targets = []
        final_outputs = []
        for idx, (inputs, label) in enumerate(dataloader):
            predicted_label = model(inputs)
            predicted_label.to(DEVICE)
            loss = loss_fn(predicted_label, torch.eye(2).to(DEVICE)[label])
            total_acc += (predicted_label.argmax(1) == label).sum().item()
            total_count += label.size(0)

            preds = predicted_label.argmax(1)
            targets = label.detach().cpu().numpy().tolist()
            output = preds.detach().cpu().numpy().tolist()
            final_targets.extend(targets)
            final_outputs.extend(output)
    
    final_outputs = np.array(final_outputs)
    final_targets = np.array(final_targets)
    final_targets = np.reshape(final_targets, (-1,))

    np.savetxt('outputs.csv', final_outputs, delimiter=',')

    f1_macro, TP, FP, FN, TN = f1_macro(final_outputs, final_targets)

    return {
        'loss': loss.item(), 
        'acc': total_acc / total_count, 
        'f1': f1_macro, 
        'TP': TP,
        'FP': FP,
        'TN': TN,
        'FN': FN
        }

In [ ]:
def train_and_evaluate():

    # Загрузка предварительно обученного токенизатора и модели BERT
    # поменяйте пути на те, которые используете
    tokenizer = transformers.BertTokenizer(
        vocab_file=VOCAB_PATH,
        truncation=True,
        max_length=512
        )
    model = transformers.BertModel.from_pretrained(
         MODEL_PATH, 
         config=config,
         ignore_mismatched_sizes=True,
    )
    # Параметры обучения
    epochs = 4
    batch_size = 20
    
    #  Загрузка данных
    df_train = pd.read_csv('./train.csv')
    df_eval = pd.read_csv('./eval.csv')
    # df_test = pd.read_csv('./test.csv')

    # Создание загрузчиков данных
    train_ds = BertDataset(df_train, tokenizer, max_length=512)
    # train_sampler = RandomSampler(train_ds,replacement=True)
    # train_sampler = SequentialSampler(train_ds)
    train_loader = DataLoader(
        dataset=train_ds,
        shuffle=True,
        # sampler=train_sampler,
        batch_size=batch_size, 
        drop_last=False
        )
    
    eval_ds = BertDataset(df_eval, tokenizer, max_length=512)
    eval_loader = DataLoader(
        dataset=eval_ds,
        batch_size=batch_size
        )
    # test_ds = BertDataset(df_test, tokenizer, max_length=512)
    # test_loader = DataLoader(
    #    dataset=test_ds,
    #    batch_size=batch_size
    #    )
    
    # Создание модели
    classifier = OperationsBERT(bert_model=model).to(DEVICE)
    total_parameters = sum([np.prod(p.size()) for p in classifier.parameters()])
    model_parameters = filter(lambda p: p.requires_grad, classifier.parameters())
    params = sum([np.prod(p.size()) for p in model_parameters])
    
    # Оптимизатор и функция потерь
    # optimizer = torch.optim.Adam(
    #    [p for p in classifier.parameters() if p.requires_grad], 
    #    learning_rate
    #    )
    optimizer = torch.optim.AdamW(
        params= model.parameters(), 
        lr=0.001,
        weight_decay=1e-6
        )
    weights = torch.tensor([0.1, 0.9]).to(DEVICE)
    pos_weight = torch.tensor([2]).to(DEVICE)
    loss_fn = nn.BCEWithLogitsLoss(
        # weight=weights,
        pos_weight=pos_weight,
        )
    
    for epoch in notebook.tqdm(range(1, epochs + 1)):
        epoch_start_time = time.time()
        train_metrics = train(
            epoch, 
            classifier, 
            train_loader, 
            loss_fn=loss_fn,
            f1_macro=f1_macro, 
            optimizer=optimizer, 
            max_steps=None
            )
        
        print('-' * 59)

        print(
            f'End of epoch {epoch} - time: {round(time.time() - epoch_start_time, 2)}s - '
            f'loss: {round(train_metrics['loss'], 4)} - accuracy: {round(train_metrics['acc'], 4)} - '
            f'f1_macro: {round(train_metrics['f1'], 4)} - TP: {train_metrics['TP']} - FP: {train_metrics['FP']} '
            f'TN: {train_metrics['TN']} - FN: {train_metrics['FN']} '
        )
        print('-' * 59)
        
        eval_metrics = evaluate(classifier, eval_loader, loss_fn=loss_fn, f1_macro=f1_macro)
        
        print('-' * 59)
        print(
            f'End of epoch {epoch} - time: {round(time.time() - epoch_start_time, 2)}s - '
            f'valid_loss: {round(eval_metrics['loss'], 4)} - valid accuracy {round(eval_metrics['acc'], 4)} '
            f'f1_macro: {round(eval_metrics['f1'], 4)} - TP: {eval_metrics['TP']} - FP: {eval_metrics['FP']} '
            f'TN: {eval_metrics['TN']} - FN: {eval_metrics['FN']} '
        )
        print("-" * 59)
    
    
    # test_metrics = evaluate(classifier, test_loader, loss_fn=loss_fn)
    
    metrics = {
        'train': train_metrics,
        'val': eval_metrics,
    #    'test': test_metrics,
    }
    
    # Сохранение модели и архитектуры в одном файле
    
    torch.save(classifier.state_dict(), './saved_model_data_4/operations_model.bin')
    tokenizer.save_pretrained('./saved_model_data_4/')
    return metrics

In [11]:
train_and_evaluate()

  0%|          | 0/4 [00:00<?, ?it/s]

Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.
/Users/alexeyfilichkin/opt/anaconda3/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:2888: FutureWarning: The `pad_to_max_length` argument is deprecated and will be removed in a future version, use `padding=True` or `padding='longest'` to pad to the longest sequence in the batch, or use `padding='max_length'` to pad to a max length. In this case, you can give a specific length with `max_length` (e.g. `max_length=45`) or leave max_length to None to pad to the maximal input size of the model (e.g. 512 for Bert).
  warnings.warn(


Epoch   1 |     0/  441 batches | accuracy    0.500 | loss    1.035 (7.610s)
Epoch   1 |    50/  441 batches | accuracy    0.882 | loss    0.505 (905.632s)
Epoch   1 |   100/  441 batches | accuracy    0.915 | loss    0.029 (935.200s)
Epoch   1 |   150/  441 batches | accuracy    0.930 | loss    0.061 (1155.042s)
Epoch   1 |   200/  441 batches | accuracy    0.940 | loss    0.059 (1120.224s)
Epoch   1 |   250/  441 batches | accuracy    0.945 | loss    0.131 (944.819s)
Epoch   1 |   300/  441 batches | accuracy    0.948 | loss    0.036 (750.362s)
Epoch   1 |   350/  441 batches | accuracy    0.950 | loss    0.372 (870.604s)
Epoch   1 |   400/  441 batches | accuracy    0.951 | loss    0.140 (1341.826s)
-----------------------------------------------------------
End of epoch 1 - time: 8621.55s - loss: 0.0492 - accuracy: 0.9517 - f1_macro: 0.9516 - TP: 4074 - FP: 93 TN: 4314 - FN: 333 
-----------------------------------------------------------
-------------------------------------------

{'train': {'loss': 0.004075671546161175,
  'acc': 0.9804855911050602,
  'f1': 0.980483371302159,
  'TP': 4274,
  'FP': 39,
  'TN': 4368,
  'FN': 133},
 'val': {'loss': 0.10502215474843979,
  'acc': 0.9004457652303121,
  'f1': 0.5459139201414338,
  'TP': 17,
  'FP': 88,
  'TN': 1801,
  'FN': 113}}